# tokens

> The styling surface: colour, space, radius, type, motion, and elevation

In [ ]:
#| default_exp tokens

Tokens are the shared values that components use for styling. Changing a token changes all the components that read it. We register each family with `Rules` in the `tokens` layer, leaving applications free to override it with ordinary CSS.

In [ ]:
from importlib.resources import files
from answerui.registry import Rules

In [ ]:
from fastcore.utils import *
from fastcore.test import *
from fasthtml.common import Div
from answerui.demo import demo


## Colour

The palette notebook produced twelve CSS custom properties: neutral and vivid ladders derived from `--hue`, with light and dark values for each. Here we give them an application default and named colour contexts.

In [ ]:
PALETTE = files('answerui').joinpath('css/palette.css').read_text()

In [ ]:
print(f'{(_ := PALETTE.splitlines())[0][:100]}…\n…\n{_[-1][:100]}…')

--n-3: light-dark(oklch(0.9646 clamp(0, 0.01797 + 0.00133*cos((1*var(--hue) - 133.91)*1deg) + 0.0007…
…
--v2: light-dark(oklch(0.3952 clamp(0, 0.11982 + 0.05278*cos((1*var(--hue) - 328.16)*1deg) + 0.02527…


In [ ]:
rungs = re.findall(r'^--([\w-]+):', PALETTE, re.M)
test_eq(len(rungs), 12)


### Application palette

We use hue 260 as the default. `color-scheme: light dark` lets the browser follow the user's preference.

In [ ]:
_hue_css = ':root { --hue: 260; color-scheme: light dark; }'

### Colour contexts

Danger, warning, success, and information each get a hue and a bare attribute. These hues are independent of the application hue.

In [ ]:
CONTEXTS = dict(danger=25, warning=75, success=145, info=230)

A custom property is substituted where it is declared, so an element that sets `--hue` inherits rungs already derived from its parent's hue. A colour context therefore needs two things: `--hue` re-pointed on its selector, and the ladder declared on that same selector. `context` records both. The palette rule is built from every context declared, whenever the stylesheet is assembled, so a later module or an application can add a context without repeating the ladder:


In [ ]:
_contexts = {}

def context(
    sel, # Selector of the context
    hue, # Its hue: a number, or an expression such as `var(--hue-danger)`
):
    "Declare `sel` as a colour context whose family derives from `hue`"
    _contexts[sel] = hue

In [ ]:
def _context_css(): return '\n'.join(f'{sel} {{ --hue: {hue}; }}' for sel,hue in _contexts.items())
def _palette_css(): return f"{_context_css()}\n{', '.join([':root', *_contexts])} {{\n{PALETTE}\n}}"

The four severity contexts are the first:

In [ ]:
for name in CONTEXTS: context(f'[{name}]', f'var(--hue-{name})')

In [ ]:
print(_context_css())

[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }


We register the application default, context hues, and palette declarations in the `tokens` layer.

In [ ]:
_hues = ' '.join(f'--hue-{name}: {hue};' for name,hue in CONTEXTS.items())
Rules('hue', _hue_css, layer='tokens', doc='The application hue and the supported colour schemes.')
Rules('contexts', f':root {{ {_hues} }}', layer='tokens', doc='The severity hues, each independently adjustable.')
Rules('palette', _palette_css, layer='tokens', doc='The neutral and vivid ladders, declared on the root and on every colour context.')


palette [tokens]: The neutral and vivid ladders, declared on the root and on every colour context.

To see the tokens, a helper binds a list of token names to one CSS property through inline styles:

In [ ]:
def swatches(prop, names, **kw):
    "Tiles binding each named token to `prop`"
    tiles = [Div(n, style=f'padding:.5rem;background:var(--n-2);font:12px monospace;{prop}:var(--{n})') for n in names]
    return Div(*tiles, style='display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)', **kw)

The ladders under the application hue, in a danger context, and as a dark island:

In [ ]:
demo(swatches('background', rungs), swatches('background', rungs, danger=True),
    Div(swatches('background', rungs), style='color-scheme:dark'))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
:root, [danger], [warning], [success], [info] {
--n-3: light-dark(oklch(0.9646 clamp(0, 0.01797 + 0.00133*cos((1*var(--hue) - 133.91)*1deg) + 0.00071*cos((2*var(--hue) - 305.0)*1deg), 0.02) var(--hue)), oklch(0.1736 0.02 var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9243 0.02 var(--hue)), oklch(0.2291 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7684 0.02 var(--hue)), oklch(0.377 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6318 0.02 var(--hue)), oklch(0.4965 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5083 0.02 var(--hue)), oklch(0.6185 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4053 0.02 var(--hue)), oklch(0.734 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.894 0.04667 var(--hue)), oklch(0.2611 clamp(0, 0.08118 + 0.03727*cos((1*var(--hue) - 324.0)*1deg) + 0.01857*cos((2*var(--hue) - 261.64)*1deg) + 0.01236*cos((3*var(--hue) - 100.29)*1deg) + 0.00447*cos((4*var(--hue) - 81.53)*1deg) + 0.00295*cos((5*var(--hue) - 330.35)*1deg) + 0.00454*cos((6*var(--hue) - 161.37)*1deg) + 0.00108*cos((7*var(--hue) - 148.64)*1deg) + 0.00158*cos((8*var(--hue) - 3.5)*1deg) + 0.00264*cos((9*var(--hue) - 235.89)*1deg) + 0.00067*cos((10*var(--hue) - 214.57)*1deg) + 0.00118*cos((11*var(--hue) - 75.82)*1deg), 0.13557) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7268 clamp(0, 0.13255 + 0.00897*cos((1*var(--hue) - 20.54)*1deg), 0.14154) var(--hue)), oklch(0.418 clamp(0, 0.12579 + 0.05419*cos((1*var(--hue) - 329.06)*1deg) + 0.02532*cos((2*var(--hue) - 268.1)*1deg) + 0.01691*cos((3*var(--hue) - 86.53)*1deg) + 0.0038*cos((4*var(--hue) - 84.49)*1deg) + 0.00355*cos((5*var(--hue) - 304.18)*1deg) + 0.00769*cos((6*var(--hue) - 156.06)*1deg) + 0.00162*cos((7*var(--hue) - 155.96)*1deg) + 0.00185*cos((8*var(--hue) - 5.46)*1deg) + 0.00344*cos((9*var(--hue) - 226.4)*1deg), 0.19199) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.5896 clamp(0, 0.1523 + 0.03949*cos((1*var(--hue) - 339.6)*1deg) + 0.01699*cos((2*var(--hue) - 266.54)*1deg) + 0.02168*cos((3*var(--hue) - 73.38)*1deg) + 0.00344*cos((4*var(--hue) - 190.05)*1deg) + 0.00323*cos((5*var(--hue) - 280.54)*1deg) + 0.00629*cos((6*var(--hue) - 140.44)*1deg) + 0.00341*cos((7*var(--hue) - 287.74)*1deg), 0.19043) var(--hue)), oklch(0.5444 clamp(0, 0.14946 + 0.04722*cos((1*var(--hue) - 335.04)*1deg) + 0.01895*cos((2*var(--hue) - 273.79)*1deg) + 0.02214*cos((3*var(--hue) - 73.51)*1deg) + 0.00176*cos((4*var(--hue) - 138.94)*1deg) + 0.00286*cos((5*var(--hue) - 295.31)*1deg) + 0.00739*cos((6*var(--hue) - 143.8)*1deg) + 0.00212*cos((7*var(--h

## Typography

Fonts and their sizes are defined here. Five sizes derive from the body size, `--text`, so changing it on `:root` scales typography without changing spacing or radius. We also declare regular and semibold weights, plus normal and tight line heights.

These are shared values, not element styles: the base layer will style native elements, while typrose will handle prose regions.

In [ ]:
_type_css = r"""
:root {
  --font: system-ui, sans-serif;
  --font-mono: ui-monospace, Menlo, Consolas, monospace;
  --text: 1rem;
  --text-1: calc(var(--text) * .875);
  --text1: calc(var(--text) * 1.25);
  --text2: calc(var(--text) * 1.5);
  --text3: calc(var(--text) * 2);
  --weight: 400; --weight1: 600;
  --leading-1: 1.25; --leading: 1.5;
}
"""

In [ ]:
Rules('type', _type_css, layer='tokens',
    doc='System font families, five sizes, regular and semibold weights, and two line heights.')

type [tokens]: System font families, five sizes, regular and semibold weights, and two line heights.

In [ ]:
demo(swatches('font-size', ['text-1', 'text', 'text1', 'text2', 'text3']))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
:root, [danger], [warning], [success], [info] {
--n-3: light-dark(oklch(0.9646 clamp(0, 0.01797 + 0.00133*cos((1*var(--hue) - 133.91)*1deg) + 0.00071*cos((2*var(--hue) - 305.0)*1deg), 0.02) var(--hue)), oklch(0.1736 0.02 var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9243 0.02 var(--hue)), oklch(0.2291 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7684 0.02 var(--hue)), oklch(0.377 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6318 0.02 var(--hue)), oklch(0.4965 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5083 0.02 var(--hue)), oklch(0.6185 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4053 0.02 var(--hue)), oklch(0.734 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.894 0.04667 var(--hue)), oklch(0.2611 clamp(0, 0.08118 + 0.03727*cos((1*var(--hue) - 324.0)*1deg) + 0.01857*cos((2*var(--hue) - 261.64)*1deg) + 0.01236*cos((3*var(--hue) - 100.29)*1deg) + 0.00447*cos((4*var(--hue) - 81.53)*1deg) + 0.00295*cos((5*var(--hue) - 330.35)*1deg) + 0.00454*cos((6*var(--hue) - 161.37)*1deg) + 0.00108*cos((7*var(--hue) - 148.64)*1deg) + 0.00158*cos((8*var(--hue) - 3.5)*1deg) + 0.00264*cos((9*var(--hue) - 235.89)*1deg) + 0.00067*cos((10*var(--hue) - 214.57)*1deg) + 0.00118*cos((11*var(--hue) - 75.82)*1deg), 0.13557) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7268 clamp(0, 0.13255 + 0.00897*cos((1*var(--hue) - 20.54)*1deg), 0.14154) var(--hue)), oklch(0.418 clamp(0, 0.12579 + 0.05419*cos((1*var(--hue) - 329.06)*1deg) + 0.02532*cos((2*var(--hue) - 268.1)*1deg) + 0.01691*cos((3*var(--hue) - 86.53)*1deg) + 0.0038*cos((4*var(--hue) - 84.49)*1deg) + 0.00355*cos((5*var(--hue) - 304.18)*1deg) + 0.00769*cos((6*var(--hue) - 156.06)*1deg) + 0.00162*cos((7*var(--hue) - 155.96)*1deg) + 0.00185*cos((8*var(--hue) - 5.46)*1deg) + 0.00344*cos((9*var(--hue) - 226.4)*1deg), 0.19199) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.5896 clamp(0, 0.1523 + 0.03949*cos((1*var(--hue) - 339.6)*1deg) + 0.01699*cos((2*var(--hue) - 266.54)*1deg) + 0.02168*cos((3*var(--hue) - 73.38)*1deg) + 0.00344*cos((4*var(--hue) - 190.05)*1deg) + 0.00323*cos((5*var(--hue) - 280.54)*1deg) + 0.00629*cos((6*var(--hue) - 140.44)*1deg) + 0.00341*cos((7*var(--hue) - 287.74)*1deg), 0.19043) var(--hue)), oklch(0.5444 clamp(0, 0.14946 + 0.04722*cos((1*var(--hue) - 335.04)*1deg) + 0.01895*cos((2*var(--hue) - 273.79)*1deg) + 0.02214*cos((3*var(--hue) - 73.51)*1deg) + 0.00176*cos((4*var(--hue) - 138.94)*1deg) + 0.00286*cos((5*var(--hue) - 295.31)*1deg) + 0.00739*cos((6*var(--hue) - 143.8)*1deg) + 0.00212*cos((7*var(--h

## Space

One scale spaces component interiors and the gaps between components. Each step derives from `--space`, so changing it on `:root` adjusts spacing across the application independently of font size. Components use the scale for internal padding; layout parents use it for gaps, rather than components giving themselves outer margins.

Note, the derived values resolve at the root before being inherited, so setting `--space` on a child does not recalculate its scale. If we want local density scopes, we can revisit this.

In [ ]:
Rules('space', r"""
:root {
  --space: 1rem;
  --space-3: calc(var(--space) * .25);
  --space-2: calc(var(--space) * .5);
  --space-1: calc(var(--space) * .75);
  --space1: calc(var(--space) * 1.5);
  --space2: calc(var(--space) * 2);
  --space3: calc(var(--space) * 3);
}
""", layer='tokens', doc='A spacing scale derived from one application-wide base value.')

space [tokens]: A spacing scale derived from one application-wide base value.

In [ ]:
demo(swatches('padding', ['space-3', 'space-2', 'space-1', 'space', 'space1', 'space2', 'space3']))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
:root, [danger], [warning], [success], [info] {
--n-3: light-dark(oklch(0.9646 clamp(0, 0.01797 + 0.00133*cos((1*var(--hue) - 133.91)*1deg) + 0.00071*cos((2*var(--hue) - 305.0)*1deg), 0.02) var(--hue)), oklch(0.1736 0.02 var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9243 0.02 var(--hue)), oklch(0.2291 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7684 0.02 var(--hue)), oklch(0.377 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6318 0.02 var(--hue)), oklch(0.4965 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5083 0.02 var(--hue)), oklch(0.6185 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4053 0.02 var(--hue)), oklch(0.734 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.894 0.04667 var(--hue)), oklch(0.2611 clamp(0, 0.08118 + 0.03727*cos((1*var(--hue) - 324.0)*1deg) + 0.01857*cos((2*var(--hue) - 261.64)*1deg) + 0.01236*cos((3*var(--hue) - 100.29)*1deg) + 0.00447*cos((4*var(--hue) - 81.53)*1deg) + 0.00295*cos((5*var(--hue) - 330.35)*1deg) + 0.00454*cos((6*var(--hue) - 161.37)*1deg) + 0.00108*cos((7*var(--hue) - 148.64)*1deg) + 0.00158*cos((8*var(--hue) - 3.5)*1deg) + 0.00264*cos((9*var(--hue) - 235.89)*1deg) + 0.00067*cos((10*var(--hue) - 214.57)*1deg) + 0.00118*cos((11*var(--hue) - 75.82)*1deg), 0.13557) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7268 clamp(0, 0.13255 + 0.00897*cos((1*var(--hue) - 20.54)*1deg), 0.14154) var(--hue)), oklch(0.418 clamp(0, 0.12579 + 0.05419*cos((1*var(--hue) - 329.06)*1deg) + 0.02532*cos((2*var(--hue) - 268.1)*1deg) + 0.01691*cos((3*var(--hue) - 86.53)*1deg) + 0.0038*cos((4*var(--hue) - 84.49)*1deg) + 0.00355*cos((5*var(--hue) - 304.18)*1deg) + 0.00769*cos((6*var(--hue) - 156.06)*1deg) + 0.00162*cos((7*var(--hue) - 155.96)*1deg) + 0.00185*cos((8*var(--hue) - 5.46)*1deg) + 0.00344*cos((9*var(--hue) - 226.4)*1deg), 0.19199) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.5896 clamp(0, 0.1523 + 0.03949*cos((1*var(--hue) - 339.6)*1deg) + 0.01699*cos((2*var(--hue) - 266.54)*1deg) + 0.02168*cos((3*var(--hue) - 73.38)*1deg) + 0.00344*cos((4*var(--hue) - 190.05)*1deg) + 0.00323*cos((5*var(--hue) - 280.54)*1deg) + 0.00629*cos((6*var(--hue) - 140.44)*1deg) + 0.00341*cos((7*var(--hue) - 287.74)*1deg), 0.19043) var(--hue)), oklch(0.5444 clamp(0, 0.14946 + 0.04722*cos((1*var(--hue) - 335.04)*1deg) + 0.01895*cos((2*var(--hue) - 273.79)*1deg) + 0.02214*cos((3*var(--hue) - 73.51)*1deg) + 0.00176*cos((4*var(--hue) - 138.94)*1deg) + 0.00286*cos((5*var(--hue) - 295.31)*1deg) + 0.00739*cos((6*var(--hue) - 143.8)*1deg) + 0.00212*cos((7*var(--h

## Radius

Three sizes derive from `--radius`, so changing it on `:root` adjusts the application's general roundness. `--radius-full` stays independent: a pill is a shape, not another step on the scale.

In [ ]:
Rules('radius', r"""
:root {
  --radius: .5rem;
  --radius-1: calc(var(--radius) * .5);
  --radius1: calc(var(--radius) * 2);
  --radius-full: 9999px;
}
""", layer='tokens', doc='Three corner radii derived from one base value, plus an independent pill radius.')

radius [tokens]: Three corner radii derived from one base value, plus an independent pill radius.

In [ ]:
demo(swatches('border-radius', ['radius-1', 'radius', 'radius1', 'radius-full']))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
:root, [danger], [warning], [success], [info] {
--n-3: light-dark(oklch(0.9646 clamp(0, 0.01797 + 0.00133*cos((1*var(--hue) - 133.91)*1deg) + 0.00071*cos((2*var(--hue) - 305.0)*1deg), 0.02) var(--hue)), oklch(0.1736 0.02 var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9243 0.02 var(--hue)), oklch(0.2291 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7684 0.02 var(--hue)), oklch(0.377 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6318 0.02 var(--hue)), oklch(0.4965 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5083 0.02 var(--hue)), oklch(0.6185 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4053 0.02 var(--hue)), oklch(0.734 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.894 0.04667 var(--hue)), oklch(0.2611 clamp(0, 0.08118 + 0.03727*cos((1*var(--hue) - 324.0)*1deg) + 0.01857*cos((2*var(--hue) - 261.64)*1deg) + 0.01236*cos((3*var(--hue) - 100.29)*1deg) + 0.00447*cos((4*var(--hue) - 81.53)*1deg) + 0.00295*cos((5*var(--hue) - 330.35)*1deg) + 0.00454*cos((6*var(--hue) - 161.37)*1deg) + 0.00108*cos((7*var(--hue) - 148.64)*1deg) + 0.00158*cos((8*var(--hue) - 3.5)*1deg) + 0.00264*cos((9*var(--hue) - 235.89)*1deg) + 0.00067*cos((10*var(--hue) - 214.57)*1deg) + 0.00118*cos((11*var(--hue) - 75.82)*1deg), 0.13557) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7268 clamp(0, 0.13255 + 0.00897*cos((1*var(--hue) - 20.54)*1deg), 0.14154) var(--hue)), oklch(0.418 clamp(0, 0.12579 + 0.05419*cos((1*var(--hue) - 329.06)*1deg) + 0.02532*cos((2*var(--hue) - 268.1)*1deg) + 0.01691*cos((3*var(--hue) - 86.53)*1deg) + 0.0038*cos((4*var(--hue) - 84.49)*1deg) + 0.00355*cos((5*var(--hue) - 304.18)*1deg) + 0.00769*cos((6*var(--hue) - 156.06)*1deg) + 0.00162*cos((7*var(--hue) - 155.96)*1deg) + 0.00185*cos((8*var(--hue) - 5.46)*1deg) + 0.00344*cos((9*var(--hue) - 226.4)*1deg), 0.19199) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.5896 clamp(0, 0.1523 + 0.03949*cos((1*var(--hue) - 339.6)*1deg) + 0.01699*cos((2*var(--hue) - 266.54)*1deg) + 0.02168*cos((3*var(--hue) - 73.38)*1deg) + 0.00344*cos((4*var(--hue) - 190.05)*1deg) + 0.00323*cos((5*var(--hue) - 280.54)*1deg) + 0.00629*cos((6*var(--hue) - 140.44)*1deg) + 0.00341*cos((7*var(--hue) - 287.74)*1deg), 0.19043) var(--hue)), oklch(0.5444 clamp(0, 0.14946 + 0.04722*cos((1*var(--hue) - 335.04)*1deg) + 0.01895*cos((2*var(--hue) - 273.79)*1deg) + 0.02214*cos((3*var(--hue) - 73.51)*1deg) + 0.00176*cos((4*var(--hue) - 138.94)*1deg) + 0.00286*cos((5*var(--hue) - 295.31)*1deg) + 0.00739*cos((6*var(--hue) - 143.8)*1deg) + 0.00212*cos((7*var(--h

## Motion

`--motion` sets the shared duration, and `--ease` the easing curve. The spin that busy controls share is defined here too, so any component can turn it. Components use the duration directly or multiply it for longer effects. Temporarily increasing `--motion` lets us inspect transitions in slow motion.

A reduced-motion preference sets the duration to zero for effects that consume it. Components must remain functional without motion and must not depend on an animation finishing to perform essential work.

In [ ]:
Rules('motion', r"""
:root {
  --motion: 150ms;
  --ease: cubic-bezier(.2, 0, 0, 1);
}
@media (prefers-reduced-motion: reduce) {
  :root { --motion: 0ms; }
}
@keyframes aui-spin { to { rotate: 1turn; } }
""", layer='tokens', doc='A shared duration and easing curve, with zero duration under reduced motion, and the spin shared by busy controls.')

motion [tokens]: A shared duration and easing curve, with zero duration under reduced motion, and the spin shared by busy controls.

## Elevation

Depth is relief under one light, from the top left. A raised surface has a light rim along its top and left edges and a dark rim along its bottom and right, plus a small lift beneath it. A well is the same lighting inverted, with a shadow inside its top edge. The rims are lighter and darker than the surface's own face, so we derive them from `--face` rather than from a rung: a rung can be lighter than the face in one scheme and darker in the other.

`--relief` controls the strength of this treatment. At `1` the rims and shadows have their full opacity, `.5` softens them, and `0` removes them. Components also use it to scale pressed displacement. It is declared on `:root` and inherited, so an application can adjust the whole page or one component without rewriting its state rules. Faces, borders, and state colours remain independent of relief.

The rims must be derived on the element whose face they describe. We declare them on every element and on `::before` and `::after`, with `--face` defaulting to the raised rung. A raised surface combines `--bevel` and `--raised`; a well combines `--bevel-in` and `--sunken`. Translucent black keeps the shadows independent of colour contexts.

`--floating` separates an overlay from the page beneath it. It stays independent of relief, so a flat theme can retain depth beneath dialogs and menus.

In [ ]:
Rules('elevation', r"""
:root {
  --face: var(--n-2); --relief: 1;
  --_shadow-color: light-dark(rgb(0 0 0 / .12), rgb(0 0 0 / .5));
  --floating: 0 4px 12px var(--_shadow-color), 0 12px 32px var(--_shadow-color);
}
*, ::before, ::after {
  --hi: oklch(from var(--face) calc(l + .16) c h / var(--relief));
  --lo: oklch(from var(--face) calc(l - .16) c h / var(--relief));
  --_relief-shadow: light-dark(rgb(0 0 0 / calc(.12 * var(--relief))), rgb(0 0 0 / calc(.5 * var(--relief))));
  --bevel: inset 1px 1px 0 var(--hi), inset -1px -1px 0 var(--lo);
  --bevel-in: inset 1px 1px 0 var(--lo), inset -1px -1px 0 var(--hi);
  --raised: 0 2px 4px var(--_relief-shadow);
  --sunken: inset 0 2px 4px var(--_relief-shadow);
}
""", layer='tokens', doc='Face-derived rims and raised and sunken shadows scaled by relief, with independent floating shadows.')

elevation [tokens]: Face-derived rims and raised and sunken shadows scaled by relief, with independent floating shadows.

In [ ]:
demo(swatches('box-shadow', ['bevel', 'raised', 'bevel-in', 'sunken', 'floating']))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
:root, [danger], [warning], [success], [info] {
--n-3: light-dark(oklch(0.9646 clamp(0, 0.01797 + 0.00133*cos((1*var(--hue) - 133.91)*1deg) + 0.00071*cos((2*var(--hue) - 305.0)*1deg), 0.02) var(--hue)), oklch(0.1736 0.02 var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9243 0.02 var(--hue)), oklch(0.2291 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7684 0.02 var(--hue)), oklch(0.377 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6318 0.02 var(--hue)), oklch(0.4965 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5083 0.02 var(--hue)), oklch(0.6185 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4053 0.02 var(--hue)), oklch(0.734 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.894 0.04667 var(--hue)), oklch(0.2611 clamp(0, 0.08118 + 0.03727*cos((1*var(--hue) - 324.0)*1deg) + 0.01857*cos((2*var(--hue) - 261.64)*1deg) + 0.01236*cos((3*var(--hue) - 100.29)*1deg) + 0.00447*cos((4*var(--hue) - 81.53)*1deg) + 0.00295*cos((5*var(--hue) - 330.35)*1deg) + 0.00454*cos((6*var(--hue) - 161.37)*1deg) + 0.00108*cos((7*var(--hue) - 148.64)*1deg) + 0.00158*cos((8*var(--hue) - 3.5)*1deg) + 0.00264*cos((9*var(--hue) - 235.89)*1deg) + 0.00067*cos((10*var(--hue) - 214.57)*1deg) + 0.00118*cos((11*var(--hue) - 75.82)*1deg), 0.13557) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7268 clamp(0, 0.13255 + 0.00897*cos((1*var(--hue) - 20.54)*1deg), 0.14154) var(--hue)), oklch(0.418 clamp(0, 0.12579 + 0.05419*cos((1*var(--hue) - 329.06)*1deg) + 0.02532*cos((2*var(--hue) - 268.1)*1deg) + 0.01691*cos((3*var(--hue) - 86.53)*1deg) + 0.0038*cos((4*var(--hue) - 84.49)*1deg) + 0.00355*cos((5*var(--hue) - 304.18)*1deg) + 0.00769*cos((6*var(--hue) - 156.06)*1deg) + 0.00162*cos((7*var(--hue) - 155.96)*1deg) + 0.00185*cos((8*var(--hue) - 5.46)*1deg) + 0.00344*cos((9*var(--hue) - 226.4)*1deg), 0.19199) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.5896 clamp(0, 0.1523 + 0.03949*cos((1*var(--hue) - 339.6)*1deg) + 0.01699*cos((2*var(--hue) - 266.54)*1deg) + 0.02168*cos((3*var(--hue) - 73.38)*1deg) + 0.00344*cos((4*var(--hue) - 190.05)*1deg) + 0.00323*cos((5*var(--hue) - 280.54)*1deg) + 0.00629*cos((6*var(--hue) - 140.44)*1deg) + 0.00341*cos((7*var(--hue) - 287.74)*1deg), 0.19043) var(--hue)), oklch(0.5444 clamp(0, 0.14946 + 0.04722*cos((1*var(--hue) - 335.04)*1deg) + 0.01895*cos((2*var(--hue) - 273.79)*1deg) + 0.02214*cos((3*var(--hue) - 73.51)*1deg) + 0.00176*cos((4*var(--hue) - 138.94)*1deg) + 0.00286*cos((5*var(--hue) - 295.31)*1deg) + 0.00739*cos((6*var(--hue) - 143.8)*1deg) + 0.00212*cos((7*var(--h

## Customization

Application overrides use the default `app` layer, leaving the library's entries intact. Set the base values on `:root` so their derived scales update too. Here we choose a green hue, larger text, tighter spacing, and squarer corners:

In [ ]:
Rules('app-theme', r"""
:root { --hue: 145; --text: 1.125rem; --space: .875rem; --radius: .25rem; }
""", doc='Application-wide colour, type size, spacing, and roundness.')

app-theme [app]: Application-wide colour, type size, spacing, and roundness.

The same swatches now show the theme:

In [ ]:
demo(swatches('background', rungs), swatches('font-size', ['text-1', 'text', 'text1', 'text2', 'text3']),
    swatches('padding', ['space-3', 'space-2', 'space-1', 'space', 'space1', 'space2', 'space3']),
    swatches('border-radius', ['radius-1', 'radius', 'radius1', 'radius-full']))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
:root, [danger], [warning], [success], [info] {
--n-3: light-dark(oklch(0.9646 clamp(0, 0.01797 + 0.00133*cos((1*var(--hue) - 133.91)*1deg) + 0.00071*cos((2*var(--hue) - 305.0)*1deg), 0.02) var(--hue)), oklch(0.1736 0.02 var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9243 0.02 var(--hue)), oklch(0.2291 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7684 0.02 var(--hue)), oklch(0.377 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6318 0.02 var(--hue)), oklch(0.4965 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5083 0.02 var(--hue)), oklch(0.6185 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4053 0.02 var(--hue)), oklch(0.734 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.894 0.04667 var(--hue)), oklch(0.2611 clamp(0, 0.08118 + 0.03727*cos((1*var(--hue) - 324.0)*1deg) + 0.01857*cos((2*var(--hue) - 261.64)*1deg) + 0.01236*cos((3*var(--hue) - 100.29)*1deg) + 0.00447*cos((4*var(--hue) - 81.53)*1deg) + 0.00295*cos((5*var(--hue) - 330.35)*1deg) + 0.00454*cos((6*var(--hue) - 161.37)*1deg) + 0.00108*cos((7*var(--hue) - 148.64)*1deg) + 0.00158*cos((8*var(--hue) - 3.5)*1deg) + 0.00264*cos((9*var(--hue) - 235.89)*1deg) + 0.00067*cos((10*var(--hue) - 214.57)*1deg) + 0.00118*cos((11*var(--hue) - 75.82)*1deg), 0.13557) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7268 clamp(0, 0.13255 + 0.00897*cos((1*var(--hue) - 20.54)*1deg), 0.14154) var(--hue)), oklch(0.418 clamp(0, 0.12579 + 0.05419*cos((1*var(--hue) - 329.06)*1deg) + 0.02532*cos((2*var(--hue) - 268.1)*1deg) + 0.01691*cos((3*var(--hue) - 86.53)*1deg) + 0.0038*cos((4*var(--hue) - 84.49)*1deg) + 0.00355*cos((5*var(--hue) - 304.18)*1deg) + 0.00769*cos((6*var(--hue) - 156.06)*1deg) + 0.00162*cos((7*var(--hue) - 155.96)*1deg) + 0.00185*cos((8*var(--hue) - 5.46)*1deg) + 0.00344*cos((9*var(--hue) - 226.4)*1deg), 0.19199) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.5896 clamp(0, 0.1523 + 0.03949*cos((1*var(--hue) - 339.6)*1deg) + 0.01699*cos((2*var(--hue) - 266.54)*1deg) + 0.02168*cos((3*var(--hue) - 73.38)*1deg) + 0.00344*cos((4*var(--hue) - 190.05)*1deg) + 0.00323*cos((5*var(--hue) - 280.54)*1deg) + 0.00629*cos((6*var(--hue) - 140.44)*1deg) + 0.00341*cos((7*var(--hue) - 287.74)*1deg), 0.19043) var(--hue)), oklch(0.5444 clamp(0, 0.14946 + 0.04722*cos((1*var(--hue) - 335.04)*1deg) + 0.01895*cos((2*var(--hue) - 273.79)*1deg) + 0.02214*cos((3*var(--hue) - 73.51)*1deg) + 0.00176*cos((4*var(--hue) - 138.94)*1deg) + 0.00286*cos((5*var(--hue) - 295.31)*1deg) + 0.00739*cos((6*var(--hue) - 143.8)*1deg) + 0.00212*cos((7*var(--h

A new colour context is one declaration. Put `project=True` on a FastHTML element to give it and its contents this purple palette:


In [ ]:
context('[project]', 300)


In [ ]:
demo(swatches('background', rungs, project=True))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
[project] { --hue: 300; }
:root, [danger], [warning], [success], [info], [project] {
--n-3: light-dark(oklch(0.9646 clamp(0, 0.01797 + 0.00133*cos((1*var(--hue) - 133.91)*1deg) + 0.00071*cos((2*var(--hue) - 305.0)*1deg), 0.02) var(--hue)), oklch(0.1736 0.02 var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9243 0.02 var(--hue)), oklch(0.2291 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7684 0.02 var(--hue)), oklch(0.377 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6318 0.02 var(--hue)), oklch(0.4965 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5083 0.02 var(--hue)), oklch(0.6185 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4053 0.02 var(--hue)), oklch(0.734 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.894 0.04667 var(--hue)), oklch(0.2611 clamp(0, 0.08118 + 0.03727*cos((1*var(--hue) - 324.0)*1deg) + 0.01857*cos((2*var(--hue) - 261.64)*1deg) + 0.01236*cos((3*var(--hue) - 100.29)*1deg) + 0.00447*cos((4*var(--hue) - 81.53)*1deg) + 0.00295*cos((5*var(--hue) - 330.35)*1deg) + 0.00454*cos((6*var(--hue) - 161.37)*1deg) + 0.00108*cos((7*var(--hue) - 148.64)*1deg) + 0.00158*cos((8*var(--hue) - 3.5)*1deg) + 0.00264*cos((9*var(--hue) - 235.89)*1deg) + 0.00067*cos((10*var(--hue) - 214.57)*1deg) + 0.00118*cos((11*var(--hue) - 75.82)*1deg), 0.13557) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7268 clamp(0, 0.13255 + 0.00897*cos((1*var(--hue) - 20.54)*1deg), 0.14154) var(--hue)), oklch(0.418 clamp(0, 0.12579 + 0.05419*cos((1*var(--hue) - 329.06)*1deg) + 0.02532*cos((2*var(--hue) - 268.1)*1deg) + 0.01691*cos((3*var(--hue) - 86.53)*1deg) + 0.0038*cos((4*var(--hue) - 84.49)*1deg) + 0.00355*cos((5*var(--hue) - 304.18)*1deg) + 0.00769*cos((6*var(--hue) - 156.06)*1deg) + 0.00162*cos((7*var(--hue) - 155.96)*1deg) + 0.00185*cos((8*var(--hue) - 5.46)*1deg) + 0.00344*cos((9*var(--hue) - 226.4)*1deg), 0.19199) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.5896 clamp(0, 0.1523 + 0.03949*cos((1*var(--hue) - 339.6)*1deg) + 0.01699*cos((2*var(--hue) - 266.54)*1deg) + 0.02168*cos((3*var(--hue) - 73.38)*1deg) + 0.00344*cos((4*var(--hue) - 190.05)*1deg) + 0.00323*cos((5*var(--hue) - 280.54)*1deg) + 0.00629*cos((6*var(--hue) - 140.44)*1deg) + 0.00341*cos((7*var(--hue) - 287.74)*1deg), 0.19043) var(--hue)), oklch(0.5444 clamp(0, 0.14946 + 0.04722*cos((1*var(--hue) - 335.04)*1deg) + 0.01895*cos((2*var(--hue) - 273.79)*1deg) + 0.02214*cos((3*var(--hue) - 73.51)*1deg) + 0.00176*cos((4*var(--hue) - 138.94)*1deg) + 0.00286*cos((5*var(--hue) - 295.31)*1deg) + 0.00739*cos((6*var(--hue) - 

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()